In [67]:
import joblib
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

from sklearn.tree import DecisionTreeClassifier

from sklearn.model_selection import (
    StratifiedKFold,
    cross_validate,
    GridSearchCV
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

In [68]:
X_train = joblib.load("../data/X_train_raw.joblib")
X_test = joblib.load("../data/X_test_raw.joblib")

y_train = joblib.load("../data/y_train.joblib")
y_test = joblib.load("../data/y_test.joblib")

In [69]:
numerical_features = [
    "no_of_dependents",
    " income_annum",
    " loan_amount",
    " loan_term",
    " cibil_score",
    " residential_assets_value",
    " commercial_assets_value",
    " luxury_assets_value",
    " bank_asset_value",
    " total_assets",
    " loan_to_income_ratio",
    " asset_to_loan_ratio",
    " asset_to_income_ratio"
]

categorical_features = [
    " education",
    " self_employed",
]

In [70]:
y_train = y_train.map({
    "Rejected": 0,
    "Approved": 1
})


y_test = y_test.map({
    "Rejected": 0,
    "Approved": 1
})

In [71]:
y_train.isnull()

1224    False
478     False
3065    False
326     False
2991    False
        ...  
23      False
233     False
3298    False
1397    False
3222    False
Name:  loan_status, Length: 3415, dtype: bool

In [72]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

In [73]:
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

In [74]:
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numerical_features),
    ("cat", categorical_pipeline, categorical_features)
])

In [75]:
dt_model = DecisionTreeClassifier(
    random_state=42
)

In [76]:
dt_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", dt_model)
])

In [77]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [78]:
dt_cv_results = cross_validate(
    dt_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring=[
        "accuracy",
        "precision",
        "recall",
        "f1",
        "roc_auc"
    ],
    return_train_score=True
)

In [79]:
print("Mean CV Accuracy :", dt_cv_results["test_accuracy"].mean())
print("Mean CV Precision:", dt_cv_results["test_precision"].mean())
print("Mean CV Recall   :", dt_cv_results["test_recall"].mean())
print("Mean CV F1       :", dt_cv_results["test_f1"].mean())
print("Mean CV ROC-AUC  :", dt_cv_results["test_roc_auc"].mean())

Mean CV Accuracy : 0.9982430453879942
Mean CV Precision: 0.9990610328638498
Mean CV Recall   : 0.9981176470588234
Mean CV F1       : 0.9985860109604854
Mean CV ROC-AUC  : 0.9982836297309621


In [53]:
print(X_train.columns.tolist())

['no_of_dependents', ' education', ' self_employed', ' income_annum', ' loan_amount', ' loan_term', ' cibil_score', ' residential_assets_value', ' commercial_assets_value', ' luxury_assets_value', ' bank_asset_value', ' total_assets', ' loan_to_income_ratio', ' asset_to_loan_ratio', ' asset_to_income_ratio']


In [80]:
print("Mean Train Accuracy :", dt_cv_results["train_accuracy"].mean())
print("Mean Train Precision:", dt_cv_results["train_precision"].mean())
print("Mean Train Recall   :", dt_cv_results["train_recall"].mean())
print("Mean Train F1       :", dt_cv_results["train_f1"].mean())
print("Mean Train ROC-AUC  :", dt_cv_results["train_roc_auc"].mean())

Mean Train Accuracy : 1.0
Mean Train Precision: 1.0
Mean Train Recall   : 1.0
Mean Train F1       : 1.0
Mean Train ROC-AUC  : 1.0
